<a href="https://colab.research.google.com/github/kmeng01/rome/blob/main/notebooks/rome.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" align="left"/></a>&nbsp;or in a local notebook.

In [ ]:
import os
if not(os.path.exists("saved_mlps")):
    os.chdir("rome")
! ls

baselines     dsets	   hparams    README.md   scripts		util
CITATION.cff  experiments  LICENSE    rome	  top_k_dict_edit-5
data	      globals.yml  notebooks  saved_mlps  top_k_dict_edit-5-20


In [6]:
IS_COLAB = False
ALL_DEPS = False

# Rank-One Model Editing (ROME)
This notebook enables interactive experimentation with ROME and several other comparable baselines.
The goal is to write new facts (e.g. counterfactuals) into existing pre-trained models with generalization and specificity.

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from util import nethook
from util.generate import generate_interactive, generate_fast

from experiments.py.demo import demo_model_editing, stop_execution

/home/jeffhe/.conda/envs/ke_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Here, you can specify a GPT model (`MODEL_NAME`).

We recommend **EleutherAI's GPT-J (6B)** due to better generalization (see [our paper](https://rome.baulab.info/) for details), but GPT-2 XL (1.5B) consumes less memory.
* `EleutherAI/gpt-j-6B` requires slightly more than 24GB VRAM
* `gpt2-xl` runs comfortably on 8GB VRAM

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = 'cpu'
print(f"Using device: {device}")

ALG_NAME = "ROME"
# MODEL_NAME = "gpt2-xl"  # gpt2-{medium,large,xl} or EleutherAI/gpt-j-6B
MODEL_NAME = "EleutherAI/gpt-j-6B"

Using device: cuda


In [7]:

model, tok = (
    AutoModelForCausalLM.from_pretrained(MODEL_NAME, low_cpu_mem_usage=IS_COLAB).to(
        device
    ),
    
    AutoTokenizer.from_pretrained(MODEL_NAME),
)
tok.pad_token = tok.eos_token
model.config

Some weights of the model checkpoint at EleutherAI/gpt-j-6B were not used when initializing GPTJForCausalLM: ['transformer.h.0.attn.bias', 'transformer.h.0.attn.masked_bias', 'transformer.h.1.attn.bias', 'transformer.h.1.attn.masked_bias', 'transformer.h.10.attn.bias', 'transformer.h.10.attn.masked_bias', 'transformer.h.11.attn.bias', 'transformer.h.11.attn.masked_bias', 'transformer.h.12.attn.bias', 'transformer.h.12.attn.masked_bias', 'transformer.h.13.attn.bias', 'transformer.h.13.attn.masked_bias', 'transformer.h.14.attn.bias', 'transformer.h.14.attn.masked_bias', 'transformer.h.15.attn.bias', 'transformer.h.15.attn.masked_bias', 'transformer.h.16.attn.bias', 'transformer.h.16.attn.masked_bias', 'transformer.h.17.attn.bias', 'transformer.h.17.attn.masked_bias', 'transformer.h.18.attn.bias', 'transformer.h.18.attn.masked_bias', 'transformer.h.19.attn.bias', 'transformer.h.19.attn.masked_bias', 'transformer.h.2.attn.bias', 'transformer.h.2.attn.masked_bias', 'transformer.h.20.attn.bi

GPTJConfig {
  "_attn_implementation_autoset": true,
  "_name_or_path": "EleutherAI/gpt-j-6B",
  "activation_function": "gelu_new",
  "architectures": [
    "GPTJForCausalLM"
  ],
  "attn_pdrop": 0.0,
  "bos_token_id": 50256,
  "embd_pdrop": 0.0,
  "eos_token_id": 50256,
  "gradient_checkpointing": false,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gptj",
  "n_embd": 4096,
  "n_head": 16,
  "n_inner": null,
  "n_layer": 28,
  "n_positions": 2048,
  "resid_pdrop": 0.0,
  "rotary": true,
  "rotary_dim": 64,
  "scale_attn_weights": true,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls_index",
  "summary_use_proj": true,
  "task_specific_params": {
    "text-generation": {
      "do_sample": true,
      "max_length": 50,
      "temperature": 1.0
    }
  },
  "tie_word_embeddings": false,
  "tokenizer_class": "GPT2Tokenizer",
  "transformers_version": "4.46.1",
  "use_cache": true,
  "voc

A requested rewrite can be specified using `request`. `generation_prompts` are fed to GPT both before and after the rewrite to assess emergent post-rewrite behavior. See the bottom of this notebook for more examples.


This cell executes the model edit.
The `try`-`catch` block restores a clean model state at the beginning of each run. `ALG_NAME` controls which algorithm is used. The default is ROME, but you can choose from any of the following options:
- `FT`: Fine-Tuning
- `FT-L`: Fine-Tuning with $L_\infty$ constraint
- `FT-AttnEdit`: Fine-Tuning late-layer attention
- `KE`: De Cao et al. Knowledge Editor
- `KE-CF`: KE trained on CounterFact
- `MEND`: Mitchell et al. Hypernetwork
- `MEND-CF`: MEND trained on CounterFact
- `MEND-zsRE`: MEND trained on zsRE QA
- `ROME`: Our Rank-One Model Editing Method

Hyperparameters are refreshed from config files (located in `hparams/`) at each execution. To modify any parameter, edit and save the respective file. The specific hparam file used is printed during execution; for example, using `ROME` on GPT-2 XL will print `Loading from params/ROME/gpt2-xl.json`.

ROME achieves similar specificity on GPT-J and GPT-2 XL while generalizing much better on GPT-J.


In [25]:
# save the reference for every layer
def cache_ref(model):
    return {i: model.transformer.h[i] for i in range(len(model.transformer.h))}

def remove_layer(model, remove_list):
    model.transformer.h = torch.nn.ModuleList(
        [layer for i, layer in enumerate(model.transformer.h) if i not in remove_list]
    )

# move the layer at from_layer to to_layer
def move_layer(model, from_layer, to_layer):
    if from_layer == to_layer:
        return
    model.transformer.h.insert(to_layer, model.transformer.h.pop(from_layer))

def restore_ref(model, ref):
    model.transformer.h = torch.nn.ModuleList([ref[i] for i in range(len(ref))])

layer_cache = cache_ref(model)


In [44]:
import torch

def top_k_next_tokens(model, tok, prompts, k=10,target_tok=None):
    # Tokenize input prompt
    inputs = tok(prompts, return_tensors="pt")

    # Move to GPU if available
    # device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # model = model.to(device)
    inputs = {k: v.to(next(model.parameters()).device) for k, v in inputs.items()}

    # Get model logits
    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits  # Shape: (batch_size, sequence_length, vocab_size)

    # Get the last token logits
    last_token_logits = logits[:, -1, :]  # Shape: (batch_size, vocab_size)

    # Get the top k token indices and probabilities
    probs = torch.softmax(last_token_logits, dim=-1)
    top_k_probs, top_k_indices = torch.topk(probs, k, dim=-1)

    # Convert token indices to actual words
    top_k_tokens = [tok.decode([idx]) for idx in top_k_indices[0].tolist()]

    # Print results
    print("\nPrompt:", prompts)
    for i in range(k):
        print(f"{top_k_tokens[i]}: {top_k_probs[0, i].item():.4f}")

        # If target_tok is provided, find its probability
    target_prob = None
    if target_tok is not None:
        target_id = tok.encode(target_tok, add_special_tokens=False)[0]  # Convert to token ID
        target_prob = probs[0, target_id].item() if target_id < probs.shape[1] else 0.0
        print(f"\nProbability of target token '{target_tok}': {target_prob:.4f}")

    # return a dictionary with the top k tokens and their probabilities
    return {top_k_tokens[i]: top_k_probs[0, i].item() for i in range(k)},target_prob





In [53]:
del_list = [17,18,19]
remove_layer(model, del_list)

save_experiment = True
log_path = "logs/deletion_log.json"

check_sanity = False
# check_sanity = True

print("\n")
target_tok = " Angela"
prob_dict,target_p = top_k_next_tokens(model, tok, "The president of Germany is named",target_tok=" Angela")


if check_sanity:
    # sanity check
    test_generation_prompts = [
        # "What is the twin city of Lyon? It is",
        "People like to travel to London to",
        "The country where the Eiffel Tower located in is famous of its",
        "The tallest building in the world is the"
    ]
    test_post_update_text = generate_fast(model, tok, test_generation_prompts, max_out_len=100)
    for generated_text in test_post_update_text:
        print("\n"+generated_text)

if save_experiment:
    # append del_list,target_p to log
    import json
    with open(log_path, "a") as f:
        json.dump({"del_list":del_list,"prob for"+target_tok :target_p, "prob_dict": prob_dict},f)
        f.write("\n")

# restore the layers
restore_ref(model, layer_cache)




Prompt: The president of Germany is named
 Angela: 0.3009
 a: 0.0379
 Chancellor: 0.0339
,: 0.0322
 ": 0.0275
 as: 0.0268
 in: 0.0259
.: 0.0197
 by: 0.0185
 after: 0.0178

Probability of target token ' Angela': 0.3009


In [ ]:
# stop_execution()

Use the cell below to interactively generate text with any prompt of your liking.

Here are some extra request/prompt combinations you can try. Simply run them before the editing cell!